# Voiceprint — Grande validation Common Voice 27.0 (préparée comme l'application)

**Où on est :** jour 7-8, priorité 1 : vérifier les réglages de la démo sur **290 personnes** au lieu de 28.

**Ce que fait ce notebook :**
1. Repart du zip audio déjà sur ton Drive (pas de retéléchargement des 28 Go).
2. Pour chacune des 290 personnes de validation : 4 clips de **test**, les autres forment l'**enrôlement** (3, 5 et 10 s de parole, en un bloc).
3. Prépare chaque voix avec **`preparation.py`**, exactement comme l'application ; tests de 3 s ou jusqu'à 10 s, propres, ambiance 10 dB ou brouhaha 10 dB (bruit ajouté AVANT le retrait des silences).
4. Calcule les empreintes ECAPA sur GPU et les sauvegarde sur ton Drive (quelques Mo).

**Avant de lancer :**
- Exécution > Modifier le type d'exécution > **GPU T4**.
- Dans **Mon Drive/voiceprint/** : `split_cv27.csv` (créé par `python src/separer_cv27.py`), `selection_cv27.csv` et `cv27_selection_audio.zip` (déjà là depuis le notebook 04).
- Dans **Mon Drive/voiceprint/code/** : une copie de tes fichiers `src/preparation.py` et `src/augmentation.py`.

Lance les cellules une par une, dans l'ordre.

In [ ]:
# 1) Réglages
import os, shutil, subprocess, time, hashlib
DOSSIER_DRIVE = "/content/drive/MyDrive/voiceprint"
TRAVAIL = "/content/travail"
GRAINE = 42
N_TESTS_PAR_LOC = 4                 # clips de test par personne ; les autres servent à l'enrôlement
DUREES_ENROL = [3, 5, 10]
TESTS = {"3s": 3, "max10s": 10}
CONDITIONS = {"propre": None, "ambiance": 10, "brouhaha": 10}
os.makedirs(TRAVAIL, exist_ok=True)

import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (Exécution > Modifier le type d'exécution > GPU T4)")

In [ ]:
# 2) Google Drive et vérification des fichiers
from google.colab import drive
drive.mount("/content/drive")
attendus = ["split_cv27.csv", "selection_cv27.csv", "cv27_selection_audio.zip",
            "code/preparation.py", "code/augmentation.py"]
manquants = [f for f in attendus if not os.path.exists(f"{DOSSIER_DRIVE}/{f}")]
if manquants:
    raise FileNotFoundError(f"Manquants dans Mon Drive/voiceprint/ : {manquants}")
print("Tous les fichiers sont présents ✅")

In [ ]:
# 3) Installation et import de LA recette de l'application (même fichier que sur ton PC)
!pip install -q "speechbrain==1.1.2"
import sys
sys.path.insert(0, f"{DOSSIER_DRIVE}/code")
import preparation, augmentation
from preparation import charger_brut, garder_parole, enrolement, test, empreinte_brute
from augmentation import bruit_rose, brouhaha, melanger
EMPREINTE_RECETTE = hashlib.md5(open(f"{DOSSIER_DRIVE}/code/preparation.py", "rb").read()).hexdigest()
print("preparation.py importé | empreinte du fichier (md5) :", EMPREINTE_RECETTE)

In [ ]:
# 4) Décompression de l'audio (environ 1 min)
import zipfile
AUDIO = f"{TRAVAIL}/audio"
if not os.path.isdir(AUDIO):
    with zipfile.ZipFile(f"{DOSSIER_DRIVE}/cv27_selection_audio.zip") as z:
        z.extractall(AUDIO)
print("Dossiers de locuteurs :", len(os.listdir(AUDIO)))

In [ ]:
# 5) Plan : enrôlement et clips de test de chaque personne de validation
import numpy as np, pandas as pd
rng = np.random.default_rng(GRAINE)
split = pd.read_csv(f"{DOSSIER_DRIVE}/split_cv27.csv", dtype={"locuteur": str})
sel = pd.read_csv(f"{DOSSIER_DRIVE}/selection_cv27.csv", dtype=str)
sel["fichier"] = AUDIO + "/" + sel["locuteur"] + "/" + sel["clip"] + ".mp3"
val_locs = sorted(split.loc[split["groupe"] == "val", "locuteur"])
train_locs = sorted(split.loc[split["groupe"] == "train", "locuteur"])
print(f"Validation : {len(val_locs)} personnes | train : {len(train_locs)} personnes")

lignes, enrol_fichiers = [], {}
for loc in val_locs:
    fichiers = sorted(sel.loc[sel["locuteur"] == loc, "fichier"])
    fichiers = list(rng.permutation(fichiers))
    tests, enrol = fichiers[:N_TESTS_PAR_LOC], fichiers[N_TESTS_PAR_LOC:]
    enrol_fichiers[loc] = enrol
    lignes += [{"locuteur": loc, "clip": os.path.basename(f)[:-4], "role": "test"} for f in tests]
    lignes += [{"locuteur": loc, "clip": os.path.basename(f)[:-4], "role": "enrolement"} for f in enrol]
info = pd.DataFrame(lignes)
assert not info.duplicated(["clip"]).any(), "Un clip sert deux fois !"
assert set(info.loc[info.role == "test", "clip"]).isdisjoint(info.loc[info.role == "enrolement", "clip"])
print(f"Clips de test : {(info.role == 'test').sum()} | clips d'enrôlement : {(info.role == 'enrolement').sum()}")

# Voix du brouhaha : 60 clips de personnes du groupe TRAIN (jamais de validation)
fichiers_train = sel.loc[sel["locuteur"].isin(train_locs), "fichier"].to_numpy()
voix = [garder_parole(charger_brut(f)) for f in rng.choice(np.sort(fichiers_train), 60, replace=False)]
print("Voix de brouhaha préparées :", len(voix))

In [ ]:
# 6) Signaux préparés comme l'application
a_faire, courts = {}, 0
t0 = time.time()
for loc in val_locs:
    bloc = np.concatenate([garder_parole(charger_brut(f)) for f in enrol_fichiers[loc]])
    for d in DUREES_ENROL:
        y = enrolement(bloc, d)
        courts += len(y) < d * 16000
        a_faire[f"enrol|{d}|{loc}"] = y

for clip, loc in info.loc[info.role == "test", ["clip", "locuteur"]].itertuples(index=False):
    brut = charger_brut(f"{AUDIO}/{loc}/{clip}.mp3")
    for cond, snr in CONDITIONS.items():
        if snr is None:
            b = brut
        else:
            bruit = bruit_rose(len(brut), rng) if cond == "ambiance" else brouhaha(len(brut), voix, rng)
            b = melanger(brut, bruit, snr)
        p = garder_parole(b)                                    # la recette de l'application, APRÈS le bruit
        for mode, duree in TESTS.items():
            a_faire[f"test|{mode}|{cond}|{clip}"] = test(p, duree)
print(f"Signaux : {len(a_faire)} | enrôlements plus courts que prévu : {courts} | {time.time() - t0:.0f} s")

In [ ]:
# 7) Empreintes ECAPA sur GPU (un signal = un appel, sans remplissage, comme l'application)
from speechbrain.inference.speaker import EncoderClassifier
modele = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="/content/modeles/ecapa",
                                        run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"})
modele.eval()
cles, emb, t0 = list(a_faire), [], time.time()
for n, k in enumerate(cles, 1):
    emb.append(empreinte_brute(modele, a_faire[k]))
    if n % 1000 == 0:
        print(f"  {n}/{len(cles)}  ({time.time() - t0:.0f} s)")
emb = np.stack(emb).astype(np.float32)

# Signal de contrôle (le même que dans le notebook 04) : vérifie la cohérence Colab / PC
g = np.random.default_rng(0)
t = np.arange(48000) / 16000
controle = empreinte_brute(modele, (0.05 * g.standard_normal(48000) + 0.3 * np.sin(2 * np.pi * 150 * t)).astype(np.float32))

np.savez(f"{DOSSIER_DRIVE}/banc_cv27val.npz", cles=np.array(cles), emb=emb, controle=controle,
         recette=np.array(EMPREINTE_RECETTE))
info.to_csv(f"{DOSSIER_DRIVE}/banc_cv27val_info.csv", index=False)
print(f"\nEmpreintes : {emb.shape} | NaN : {int(np.isnan(emb).sum())} | {time.time() - t0:.0f} s")
print("Sur ton Drive : banc_cv27val.npz et banc_cv27val_info.csv  ->  à copier dans data/cv27/ sur ton PC")